# NB05 — Feature Engineering for Documentation Gap Analysis

In this notebook, we create derived metrics that capture documentation quality signals from the master hospital dataset. These engineered features will enable us to identify patterns between documentation practices and hospital characteristics.

In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

# Setup
notebook_dir = Path.cwd()
data_input = notebook_dir / '../../data/outputs/nb04_merged/master_hospital_dataset.csv'
data_output_dir = notebook_dir / '../../data/outputs/nb05_features'
data_output_dir.mkdir(parents=True, exist_ok=True)
data_output = data_output_dir / 'hospital_features.csv'

# Load master dataset
df = pd.read_csv(data_input, dtype={'ccn': str})
print(f"Master dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Master dataset shape: (3280, 36)
Columns: ['ccn', 'hospital_name', 'city', 'state', 'zip_code', 'county', 'beds', 'bed_size_tier', 'ownership', 'ownership_category', 'is_teaching', 'is_urban', 'census_region', 'peer_group', 'overall_rating', 'cmi', 'wage_index', 'resident_to_bed_ratio', 'dsh_pct', 'staffed_beds_cms', 'medicare_discharges_impact', 'total_charges', 'total_payments', 'total_medicare_payments', 'avg_payment_per_discharge', 'charge_to_payment_ratio', 'distinct_drgs', 'medicare_discharges_puf', 'with_CC', 'with_MCC', 'without_CC', 'total_cc_discharges', 'pct_without_CC', 'pct_with_CC', 'pct_with_MCC', 'data_completeness']


## Severity Mix Features

Hospitals with poor documentation will have a lower share of MCC (Major Comorbidity/Complexity) cases relative to peers. These features capture the composition of diagnoses by severity level, which reflects documentation quality and case complexity.

In [2]:
# Create severity mix features
# Handle missing values in severity percentages
df['pct_with_MCC'] = df['pct_with_MCC'].fillna(0)
df['pct_with_CC'] = df['pct_with_CC'].fillna(0)
df['pct_without_CC'] = df['pct_without_CC'].fillna(0)

# Normalize to 0-1 scale
df['mcc_ratio'] = df['pct_with_MCC'] / 100
df['cc_ratio'] = df['pct_with_CC'] / 100
df['low_severity_ratio'] = df['pct_without_CC'] / 100

# Weighted severity score (0-1 scale)
df['severity_index'] = (df['pct_with_MCC'] * 2 + df['pct_with_CC'] * 1 + df['pct_without_CC'] * 0) / 200

# Print distribution stats
severity_features = ['mcc_ratio', 'cc_ratio', 'low_severity_ratio', 'severity_index']
print("Severity Mix Features Distribution:")
print(df[severity_features].describe())

Severity Mix Features Distribution:
         mcc_ratio     cc_ratio  low_severity_ratio  severity_index
count  3280.000000  3280.000000         3280.000000     3280.000000
mean      0.638621     0.174922            0.045298        0.726082
std       0.317741     0.142131            0.134616        0.329740
min       0.000000     0.000000            0.000000        0.000000
25%       0.614547     0.000000            0.000000        0.773754
50%       0.724918     0.196429            0.000000        0.849126
75%       0.833718     0.268808            0.042900        0.913573
max       1.000000     1.000000            1.000000        1.000000


## CMI-Based Features

Case Mix Index (CMI) reflects the relative costliness of cases treated. We create features that normalize and contextualize CMI to identify hospitals with unusual case complexity patterns.

In [3]:
# Create CMI-based features
# CMI percentile: rank-based percentile across all hospitals
df['cmi_percentile'] = df['cmi'].rank(pct=True) * 100

# Log transform for normalization (handle 0 values)
df['log_cmi'] = np.log(df['cmi'].replace(0, np.nan)).fillna(0)

# CMI per bed: intensity measure
# Handle division by zero
df['cmi_per_bed'] = df['cmi'] / df['beds']
df['cmi_per_bed'] = df['cmi_per_bed'].replace([np.inf, -np.inf], np.nan).fillna(0)

# Print stats
cmi_features = ['cmi_percentile', 'log_cmi', 'cmi_per_bed']
print("CMI Features Distribution:")
print(df[cmi_features].describe())

CMI Features Distribution:
       cmi_percentile      log_cmi  cmi_per_bed
count     3060.000000  3280.000000  3280.000000
mean        50.016340     0.507671     0.018012
std         28.872229     0.256202     0.040561
min          0.032680    -0.424189     0.000000
25%         25.024510     0.384973     0.004801
50%         50.016340     0.517096     0.008408
75%         75.008170     0.655848     0.016328
max        100.000000     1.586046     0.787067


## Payment Efficiency Features

Payment-based features capture the financial dimensions of hospital operations, reflecting revenue intensity and payment patterns.

In [4]:
# Create payment-based features
# Payment per discharge (already exists as avg_payment_per_discharge)
df['payment_per_discharge'] = df['avg_payment_per_discharge']

# Log total payments for normalization
df['log_total_payments'] = np.log(df['total_payments'].replace(0, np.nan)).fillna(0)

# Payment per bed: revenue intensity per bed
df['payment_per_bed'] = df['total_payments'] / df['beds']
df['payment_per_bed'] = df['payment_per_bed'].replace([np.inf, -np.inf], np.nan).fillna(0)

# Print stats
payment_features = ['payment_per_discharge', 'log_total_payments', 'payment_per_bed']
print("Payment Efficiency Features Distribution:")
print(df[payment_features].describe())

Payment Efficiency Features Distribution:
       payment_per_discharge  log_total_payments  payment_per_bed
count            2880.000000         3280.000000     3.280000e+03
mean            15630.906505           14.208966     8.394484e+04
std              7814.517373            5.507087     9.723684e+04
min              5542.323529            0.000000     0.000000e+00
25%             11387.366971           14.414457     1.975605e+04
50%             13562.938717           16.022930     5.559361e+04
75%             17147.278131           17.211456     1.149892e+05
max            126309.062325           20.727588     1.163944e+06


## Hospital Complexity Indicators

Categorical and binary features that capture hospital structure, teaching status, and safety-net mission.

In [5]:
# Create hospital complexity indicators

# DRG diversity (rename for clarity)
df['drg_diversity'] = df['distinct_drgs']

# Size indicator
df['is_large'] = (df['beds'] >= 400).astype(int)

# Safety-net indicator: disproportionate share hospitals
df['is_safety_net'] = (df['dsh_pct'] > 0.25).astype(int)

# Teaching intensity: categorize by resident to bed ratio
def categorize_teaching(ratio):
    if pd.isna(ratio):
        return 'unknown'
    elif ratio == 0:
        return 'non_teaching'
    elif ratio < 0.25:
        return 'minor_teaching'
    else:
        return 'major_teaching'

df['teaching_intensity'] = df['resident_to_bed_ratio'].apply(categorize_teaching)

# Print crosstabs
print("Hospital Size Distribution:")
print(df['is_large'].value_counts())
print("\nSafety-Net Status:")
print(df['is_safety_net'].value_counts())
print("\nTeaching Intensity:")
print(df['teaching_intensity'].value_counts())
print("\nDRG Diversity Stats:")
print(df['drg_diversity'].describe())

Hospital Size Distribution:
is_large
0    2602
1     678
Name: count, dtype: int64

Safety-Net Status:
is_safety_net
1    1911
0    1369
Name: count, dtype: int64

Teaching Intensity:
teaching_intensity
non_teaching      1811
minor_teaching     841
major_teaching     408
unknown            220
Name: count, dtype: int64

DRG Diversity Stats:
count    2880.000000
mean       50.662847
std        53.181430
min         1.000000
25%        10.000000
50%        33.000000
75%        73.000000
max       389.000000
Name: drg_diversity, dtype: float64


In [6]:
# Final feature summary
engineered_cols = [
    'mcc_ratio', 'cc_ratio', 'low_severity_ratio', 'severity_index',
    'cmi_percentile', 'log_cmi', 'cmi_per_bed',
    'payment_per_discharge', 'log_total_payments', 'payment_per_bed',
    'drg_diversity', 'is_large', 'is_safety_net', 'teaching_intensity'
]

print(f"Dataset shape after feature engineering: {df.shape}")
print(f"\nNew engineered features: {engineered_cols}")
print(f"\nMissing values in engineered features:")
print(df[engineered_cols].isnull().sum())
print(f"\nAll columns: {list(df.columns)}")

Dataset shape after feature engineering: (3280, 50)

New engineered features: ['mcc_ratio', 'cc_ratio', 'low_severity_ratio', 'severity_index', 'cmi_percentile', 'log_cmi', 'cmi_per_bed', 'payment_per_discharge', 'log_total_payments', 'payment_per_bed', 'drg_diversity', 'is_large', 'is_safety_net', 'teaching_intensity']

Missing values in engineered features:
mcc_ratio                  0
cc_ratio                   0
low_severity_ratio         0
severity_index             0
cmi_percentile           220
log_cmi                    0
cmi_per_bed                0
payment_per_discharge    400
log_total_payments         0
payment_per_bed            0
drg_diversity            400
is_large                   0
is_safety_net              0
teaching_intensity         0
dtype: int64

All columns: ['ccn', 'hospital_name', 'city', 'state', 'zip_code', 'county', 'beds', 'bed_size_tier', 'ownership', 'ownership_category', 'is_teaching', 'is_urban', 'census_region', 'peer_group', 'overall_rating', 'cmi'

In [7]:
# Save to output
df.to_csv(data_output, index=False)
print(f"Feature engineering complete!")
print(f"Saved {df.shape[0]} hospitals with {df.shape[1]} features to {data_output}")
print(f"\nNew engineered features created:")
for col in engineered_cols:
    print(f"  - {col}")

Feature engineering complete!
Saved 3280 hospitals with 50 features to /Users/trinidadcisneros/Documents/Development/Coding/bitterscientist.com/bitterscientist.com/folders/ds_blogs/projects/smarterdx_documentation_gap/notebooks/2_hospital_analysis/../../data/outputs/nb05_features/hospital_features.csv

New engineered features created:
  - mcc_ratio
  - cc_ratio
  - low_severity_ratio
  - severity_index
  - cmi_percentile
  - log_cmi
  - cmi_per_bed
  - payment_per_discharge
  - log_total_payments
  - payment_per_bed
  - drg_diversity
  - is_large
  - is_safety_net
  - teaching_intensity
